# 🌍 GbeTo : Traducteur Éwé ↔ Français — Notebook Principal

**Auteur** : Kodjo Jean DEGBEVI  
**Modèle** : `facebook/nllb-200-distilled-600M` (fine-tuning)  
**Données** : AfroLingu-MT (ACL 2024) + MAFAND (NAACL 2022)  
**Repo** : [github.com/kjd-dktech/ewe-french_translator](https://github.com/kjd-dktech/ewe-french_translator)

---

## Structure du notebook

| Section | Contenu |
|---------|----------|
| **0** | Configuration de l'environnement Colab |
| **1** | Pipeline de données |
| **2** | Analyse exploratoire (EDA) |
| **3** | Entraînement |
| **4** | Évaluation et résultats |
| **5** | Error analysis |
| **6** | Démonstration interactive |

---
## Section 0 — Configuration de l'environnement

In [11]:
# ── 0.1 Vérification du GPU ──────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU : {gpu_name}")
    print(f"   VRAM : {vram_gb:.1f} GB")
else:
    print("⚠️  Aucun GPU détecté — activer le GPU dans Exécution → Modifier le type d'exécution")

✅ GPU : Tesla T4
   VRAM : 15.6 GB


In [ ]:
# ── 0.2 Montage Google Drive ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH = '/content/drive/MyDrive/ewe-french_translator'
print(f"Chemin du projet : {PROJECT_PATH}")


"""# ── 0.2 Connexion du workspace local : cas où on a une connexion au serveur colab depuis le local ─────────────────────────────────────────────────
!pip install colab_ssh -q
import getpass
from colab_ssh import launch_ssh

# Une fenêtre de saisie apparaîtra dans VS Code
NGROK_TOKEN = getpass.getpass("Collez votre token Ngrok ici :")

launch_ssh(NGROK_TOKEN, "GbeTo - Colab")
print("✅ Connexion SSH établie.")"""


Chemin du projet : /run/media/kjd/working/projects/IA/NLP_Ewe-French/NLP_1


In [ ]:
# ── 0.3 Positionnement dans le dossier projet ────────────────────────────────
import os
os.chdir(PROJECT_PATH)

import subprocess
result = subprocess.run(['pwd'], capture_output=True, text=True)
print(f"Répertoire courant : {result.stdout.strip()}")

FileNotFoundError: [Errno 2] No such file or directory: '/run/media/kjd/working/projects/IA/NLP_Ewe-French/NLP_1'

In [ ]:
# ── 0.4 Installation des dépendances ────────────────────────────────────────
!pip install -r requirements.txt -q
print("✅ Dépendances installées.")

In [ ]:
# ── 0.5 Authentification — HuggingFace et W&B ───────────────────────────────
#
from google.colab import userdata
import os

# HuggingFace (requis pour AfroLingu-MT gated)
HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN

from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print("✅ HuggingFace : authentifié.")

# Weights & Biases
WANDB_API_KEY = userdata.get('WANDB_API_KEY')
os.environ['WANDB_API_KEY'] = WANDB_API_KEY

import wandb
wandb.login(key=WANDB_API_KEY)
print("✅ W&B : authentifié.")


# ====================================================================================
# ====================================================================================
# ====================================================================================

"""
import os
from dotenv import load_dotenv
from huggingface_hub import login
import wandb

# Charger les variables depuis le fichier .env
load_dotenv()

# HuggingFace
HF_TOKEN = os.getenv('HF_TOKEN')
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("✅ HuggingFace : authentifié.")
else:
    print("❌ Erreur : HF_TOKEN introuvable dans le fichier .env")

# Weights & Biases
WANDB_API_KEY = os.getenv('WANDB_API_KEY')
if WANDB_API_KEY:
    wandb.login(key=WANDB_API_KEY)
    print("✅ W&B : authentifié.")
else:
    print("❌ Erreur : WANDB_API_KEY introuvable dans le fichier .env")"""

In [ ]:
# ── 0.6 Gestion de la continuité W&B après coupure Colab ────────────────────
#
# PREMIER LANCEMENT :
#   On laisse WANDB_RUN_ID = None
#   W&B créera une nouvelle run et affichera son ID dans les logs
#
# REPRISE APRÈS COUPURE :
#   On renseigne WANDB_RUN_ID avec l'ID de la run précédente
#   Les nouvelles métriques s'ajouteront à la même run sur wandb.ai
#
WANDB_RUN_ID = None

if WANDB_RUN_ID:
    os.environ['WANDB_RUN_ID'] = WANDB_RUN_ID
    os.environ['WANDB_RESUME'] = 'must'
    print(f"✅ W&B : reprise de la run {WANDB_RUN_ID}")
else:
    os.environ.pop('WANDB_RUN_ID', None)
    os.environ.pop('WANDB_RESUME', None)
    print("ℹ️  W&B : nouvelle run (premier lancement)")

---
## Section 1 — Pipeline de données

In [ ]:
# ── 1.1 Téléchargement et fusion des datasets ────────────────────────────────
#
# Sources :
#   - UBC-NLP/AfroLingu-MT (ACL 2024) — gated, nécessite HF_TOKEN
#   - masakhane/mafand (NAACL 2022) — public
#
# Sorties : data/raw/merged_<split>.csv
#
!python -m src.data.download --output_dir data/raw

In [ ]:
# ── 1.2 Nettoyage et filtrage ────────────────────────────────────────────────
#
# Filtres appliqués dans l'ordre :
#   1. Normalisation Unicode NFC
#   2. Déduplication
#   3. Longueur : 3 ≤ mots ≤ 150
#   4. Ratio source/target : 0.2 ≤ ratio ≤ 5.0
#
!python -m src.data.filter --input_dir data/raw --output_dir data/processed

In [ ]:
# ── 1.3 Finalisation des splits ──────────────────────────────────────────────
#
# Shuffle reproductible avec seed=42
# Sorties : data/processed/train.csv, val.csv, test.csv
#
!python -m src.data.split --input_dir data/processed --output_dir data/processed

In [ ]:
# ── 1.4 Vérification des fichiers produits ───────────────────────────────────
import pandas as pd
from pathlib import Path

splits = {'train': 'data/processed/train.csv',
          'val':   'data/processed/val.csv',
          'test':  'data/processed/test.csv'}

print(f"{'Split':<8} {'Total':>8} {'ewe-fra':>10} {'fra-ewe':>10} {'afrolingu':>12} {'mafand':>10}")
print("-" * 62)

for name, path in splits.items():
    df = pd.read_csv(path)
    ewe_fra   = (df['direction'] == 'ewe-fra').sum()
    fra_ewe   = (df['direction'] == 'fra-ewe').sum()
    afrolingu = (df['origin'] == 'afrolingu').sum()
    mafand    = (df['origin'] == 'mafand').sum()
    print(f"{name:<8} {len(df):>8,} {ewe_fra:>10,} {fra_ewe:>10,} {afrolingu:>12,} {mafand:>10,}")

---
## Section 2 — Analyse exploratoire (EDA)

In [ ]:
# ── 2.0 Imports pour la visualisation ───────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd

plt.rcParams.update({
    'figure.dpi':      120,
    'font.size':       11,
    'axes.spines.top':   False,
    'axes.spines.right': False,
})

df_train = pd.read_csv('data/processed/train.csv')
df_val   = pd.read_csv('data/processed/val.csv')
df_test  = pd.read_csv('data/processed/test.csv')
print("Données chargées pour l'EDA.")

In [ ]:
# ── 2.1 Distribution des longueurs (en mots) ────────────────────────────────
df_train['src_len'] = df_train['source'].apply(lambda x: len(str(x).split()))
df_train['tgt_len'] = df_train['target'].apply(lambda x: len(str(x).split()))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, col, label, color in zip(
    axes,
    ['src_len', 'tgt_len'],
    ['Source (mots)', 'Target (mots)'],
    ['#4C72B0', '#DD8452']
):
    ax.hist(df_train[col], bins=40, color=color, alpha=0.85, edgecolor='white')
    ax.axvline(df_train[col].mean(),   color='black', linestyle='--',
               linewidth=1.2, label=f"Moyenne : {df_train[col].mean():.1f}")
    ax.axvline(df_train[col].median(), color='gray',  linestyle=':',
               linewidth=1.2, label=f"Médiane : {df_train[col].median():.1f}")
    ax.set_xlabel(label)
    ax.set_ylabel('Fréquence')
    ax.set_title(f'Distribution — {label}')
    ax.legend()

plt.suptitle('Distribution des longueurs — Train set', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/eda_length_distribution.png', bbox_inches='tight')
plt.show()

print(f"\nSource — moy: {df_train['src_len'].mean():.1f}  "
      f"méd: {df_train['src_len'].median():.0f}  "
      f"max: {df_train['src_len'].max()}")
print(f"Target — moy: {df_train['tgt_len'].mean():.1f}  "
      f"méd: {df_train['tgt_len'].median():.0f}  "
      f"max: {df_train['tgt_len'].max()}")

In [ ]:
# ── 2.2 Répartition par origine et direction ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Répartition par origine
origin_counts = df_train['origin'].value_counts()
axes[0].bar(origin_counts.index, origin_counts.values,
            color=['#4C72B0', '#DD8452'], edgecolor='white')
axes[0].set_title('Répartition par origine — Train')
axes[0].set_ylabel('Nombre de paires')
for i, v in enumerate(origin_counts.values):
    axes[0].text(i, v + 50, f'{v:,}', ha='center', fontsize=10)

# Répartition par direction
dir_counts = df_train['direction'].value_counts()
axes[1].bar(dir_counts.index, dir_counts.values,
            color=['#55A868', '#C44E52'], edgecolor='white')
axes[1].set_title('Répartition par direction — Train')
axes[1].set_ylabel('Nombre de paires')
for i, v in enumerate(dir_counts.values):
    axes[1].text(i, v + 50, f'{v:,}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('outputs/eda_origin_direction.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 2.3 Exemples de paires du dataset ───────────────────────────────────────
print("=" * 70)
print("EXEMPLES DE PAIRES — AfroLingu-MT")
print("=" * 70)
sample_afro = df_train[df_train['origin'] == 'afrolingu'].sample(3, random_state=42)
for _, row in sample_afro.iterrows():
    print(f"\n  Direction : {row['direction']}")
    print(f"  Source    : {row['source']}")
    print(f"  Target    : {row['target']}")

print("\n" + "=" * 70)
print("EXEMPLES DE PAIRES — MAFAND")
print("=" * 70)
sample_mafand = df_train[df_train['origin'] == 'mafand'].sample(3, random_state=42)
for _, row in sample_mafand.iterrows():
    print(f"\n  Direction : {row['direction']}")
    print(f"  Source    : {row['source']}")
    print(f"  Target    : {row['target']}")

---
## Section 3 — Entraînement

**Comportement automatique :**
- Si `outputs/checkpoints/` contient des checkpoints → **reprise** depuis le dernier état
- Sinon → **nouveau départ**

**Suivi en temps réel :** [wandb.ai](https://wandb.ai) → projet `GbeTo_1`

In [ ]:
# ── 3.1 Lancement de l'entraînement ──────────────────────────────────────────
#
# Le script détecte automatiquement si c'est un nouveau départ ou une reprise.
# Les métriques sont envoyées en temps réel sur W&B.
#
!bash scripts/train.sh

In [ ]:
# ── 3.2 Récupération du W&B Run ID (à faire après le premier lancement) ──────
#
# C'est l'ID affiché dans les logs W&B que je copie et colle dans la cellule 0.6
# pour assurer la continuité de la run en cas de coupure Colab.
#
# Exemple de log W&B :
#   wandb: Run data is saved locally in /content/.../wandb/run-20240315_143022-x7k2m9p1
#   → Run ID = x7k2m9p1
#
print("ℹ️  Copier le Run ID W&B affiché dans les logs ci-dessus")
print("   et le renseigner dans WANDB_RUN_ID (cellule 0.6) avant toute reprise.")

---
## Section 4 — Évaluation et résultats

In [ ]:
# ── 4.1 Chargement des métriques finales ────────────────────────────────────
import json
from pathlib import Path

metrics_path = Path('outputs/final_metrics.json')

if metrics_path.exists():
    with open(metrics_path) as f:
        metrics = json.load(f)

    print("=" * 50)
    print("MÉTRIQUES FINALES")
    print("=" * 50)
    print(f"  Modèle      : {metrics.get('model', 'N/A')}")
    print(f"  BLEU (val)  : {metrics.get('bleu_val', 0):.2f}")
    print(f"  chrF (val)  : {metrics.get('chrf_val', 0):.2f}")
    print(f"  Loss (val)  : {metrics.get('loss_val', 0):.4f}")
    print(f"  Train loss  : {metrics.get('train_loss', 0):.4f}")
    print()
    print("Hyperparamètres :")
    for k, v in metrics.get('hyperparameters', {}).items():
        print(f"  {k:<35} : {v}")
else:
    print("⚠️  outputs/final_metrics.json introuvable — lancer l'entraînement d'abord.")

In [ ]:
# ── 4.2 Comparaison baseline NLLB vs modèle fine-tuné ───────────────────────
#
# Évaluation du modèle baseline (sans fine-tuning) sur le test set
# pour mesurer le gain apporté par le fine-tuning.
#
import torch
import pandas as pd
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from src.evaluate.metrics import compute_bleu, compute_chrf
from tqdm import tqdm

def evaluate_model(model_path: str, df_test: pd.DataFrame,
                   label: str, n_samples: int = 200) -> dict:
    """
    Évalue un modèle sur un sous-ensemble du test set.
    n_samples=200 pour limiter le temps d'évaluation du baseline.
    """
    print(f"\nChargement : {label} ...")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model     = AutoModelForSeq2SeqLM.from_pretrained(model_path)
    device    = 'cuda' if torch.cuda.is_available() else 'cpu'
    model     = model.to(device).eval()

    # Sous-ensemble stratifié par direction
    df_sample = df_test.groupby('direction', group_keys=False).apply(
        lambda x: x.sample(min(n_samples // 2, len(x)), random_state=42)
    ).reset_index(drop=True)

    lang_map = {'ewe-fra': ('ewe_Latn', 'fra_Latn'),
                'fra-ewe': ('fra_Latn', 'ewe_Latn')}

    hypotheses, references = [], []

    for _, row in tqdm(df_sample.iterrows(), total=len(df_sample),
                       desc=f"  Génération [{label}]"):
        src_lang, tgt_lang = lang_map.get(row['direction'],
                                          ('ewe_Latn', 'fra_Latn'))
        tokenizer.src_lang = src_lang
        inputs = tokenizer(row['source'], return_tensors='pt',
                           max_length=128, truncation=True).to(device)
        forced_bos = tokenizer.convert_tokens_to_ids(tgt_lang)

        with torch.no_grad():
            out = model.generate(**inputs, forced_bos_token_id=forced_bos,
                                 num_beams=4, max_length=256)
        hyp = tokenizer.decode(out[0], skip_special_tokens=True).strip()
        hypotheses.append(hyp)
        references.append(row['target'])

    return {
        'label': label,
        'bleu':  compute_bleu(hypotheses, references),
        'chrf':  compute_chrf(hypotheses, references),
        'n':     len(df_sample),
    }

df_test = pd.read_csv('data/processed/test.csv')

# Baseline
results_baseline = evaluate_model(
    'facebook/nllb-200-distilled-600M', df_test, 'Baseline NLLB'
)

# Modèle fine-tuné
results_finetuned = evaluate_model(
    'outputs/checkpoints/best_model', df_test, 'Fine-tuné'
)

print("\n" + "=" * 50)
print("COMPARAISON BASELINE vs FINE-TUNÉ")
print("=" * 50)
print(f"{'Modèle':<25} {'BLEU':>8} {'chrF':>8} {'Δ BLEU':>10}")
print("-" * 55)
for r in [results_baseline, results_finetuned]:
    delta = f"+{r['bleu'] - results_baseline['bleu']:.2f}" if r['label'] != 'Baseline NLLB' else "—"
    print(f"{r['label']:<25} {r['bleu']:>8.2f} {r['chrf']:>8.2f} {delta:>10}")

In [ ]:
# ── 4.3 Graphique de comparaison ─────────────────────────────────────────────
labels  = ['Baseline NLLB', 'Fine-tuné']
bleu_scores = [results_baseline['bleu'], results_finetuned['bleu']]
chrf_scores = [results_baseline['chrf'], results_finetuned['chrf']]

x     = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width/2, bleu_scores, width, label='BLEU',
               color='#4C72B0', edgecolor='white')
bars2 = ax.bar(x + width/2, chrf_scores, width, label='chrF',
               color='#DD8452', edgecolor='white')

ax.set_ylabel('Score')
ax.set_title('Baseline NLLB vs Modèle Fine-tuné\n(éwé ↔ français, test set)',
             fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()
ax.set_ylim(0, max(chrf_scores) * 1.2)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('outputs/comparison_baseline_finetuned.png', bbox_inches='tight')
plt.show()

---
## Section 5 — Error Analysis

Identification et analyse des traductions les plus mauvaises pour comprendre
les limites du modèle et les axes d'amélioration.

In [ ]:
# ── 5.1 Génération des traductions sur le test set complet ──────────────────
import torch
import pandas as pd
from pathlib import Path
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from src.evaluate.metrics import compute_bleu, compute_chrf
from tqdm import tqdm
import sacrebleu

print("Chargement du modèle fine-tuné ...")
tokenizer = AutoTokenizer.from_pretrained('outputs/checkpoints/best_model')
model     = AutoModelForSeq2SeqLM.from_pretrained('outputs/checkpoints/best_model')
device    = 'cuda' if torch.cuda.is_available() else 'cpu'
model     = model.to(device).eval()

df_test   = pd.read_csv('data/processed/test.csv')
lang_map  = {'ewe-fra': ('ewe_Latn', 'fra_Latn'),
             'fra-ewe': ('fra_Latn', 'ewe_Latn')}

results = []
for _, row in tqdm(df_test.iterrows(), total=len(df_test), desc="Génération"):
    src_lang, tgt_lang = lang_map.get(row['direction'], ('ewe_Latn', 'fra_Latn'))
    tokenizer.src_lang = src_lang
    inputs = tokenizer(row['source'], return_tensors='pt',
                       max_length=128, truncation=True).to(device)
    forced_bos = tokenizer.convert_tokens_to_ids(tgt_lang)

    with torch.no_grad():
        out = model.generate(**inputs, forced_bos_token_id=forced_bos,
                             num_beams=4, max_length=256)
    hypothesis = tokenizer.decode(out[0], skip_special_tokens=True).strip()

    # Score BLEU individuel (sentence-level)
    sent_bleu = sacrebleu.sentence_bleu(
        hypothesis, [row['target']], tokenize='flores101'
    ).score

    results.append({
        'source':     row['source'],
        'reference':  row['target'],
        'hypothesis': hypothesis,
        'direction':  row['direction'],
        'origin':     row['origin'],
        'bleu_sent':  round(sent_bleu, 2),
    })

df_results = pd.DataFrame(results)
df_results.to_csv('outputs/error_analysis/test_predictions.csv',
                  index=False, encoding='utf-8')
print(f"\n✅ {len(df_results):,} prédictions sauvegardées.")

In [ ]:
# ── 5.2 Les 20 pires traductions (score BLEU individuel le plus bas) ─────────
df_worst = df_results.nsmallest(20, 'bleu_sent')

print("=" * 75)
print("20 PIRES TRADUCTIONS (BLEU sentence-level)")
print("=" * 75)

for i, (_, row) in enumerate(df_worst.iterrows(), 1):
    print(f"\n[{i:02d}] Direction : {row['direction']}  |  BLEU : {row['bleu_sent']:.1f}  |  Origine : {row['origin']}")
    print(f"     Source     : {row['source'][:100]}")
    print(f"     Référence  : {row['reference'][:100]}")
    print(f"     Hypothèse  : {row['hypothesis'][:100]}")

df_worst.to_csv('outputs/error_analysis/worst_20_predictions.csv',
                index=False, encoding='utf-8')
print("\n✅ Sauvegardé : outputs/error_analysis/worst_20_predictions.csv")

In [ ]:
# ── 5.3 Distribution des scores BLEU individuels ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Distribution globale
axes[0].hist(df_results['bleu_sent'], bins=30,
             color='#4C72B0', alpha=0.85, edgecolor='white')
axes[0].axvline(df_results['bleu_sent'].mean(), color='red',
                linestyle='--', label=f"Moyenne : {df_results['bleu_sent'].mean():.1f}")
axes[0].set_xlabel('BLEU sentence-level')
axes[0].set_ylabel('Fréquence')
axes[0].set_title('Distribution BLEU — Test set complet')
axes[0].legend()

# Par direction
for direction, color in [('ewe-fra', '#55A868'), ('fra-ewe', '#C44E52')]:
    subset = df_results[df_results['direction'] == direction]['bleu_sent']
    axes[1].hist(subset, bins=25, alpha=0.65, label=direction,
                 color=color, edgecolor='white')

axes[1].set_xlabel('BLEU sentence-level')
axes[1].set_ylabel('Fréquence')
axes[1].set_title('Distribution BLEU par direction')
axes[1].legend()

plt.suptitle('Analyse des scores BLEU individuels', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/error_analysis/bleu_distribution.png', bbox_inches='tight')
plt.show()

print("\nStatistiques par direction :")
print(df_results.groupby('direction')['bleu_sent'].agg(['mean','median','std']).round(2))

---
## Section 6 — Démonstration interactive

In [ ]:
# ── 6.1 Traduction interactive ───────────────────────────────────────────────
#
# Modifie le texte et la direction ci-dessous pour tester le modèle.
#
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Chargement (skip si déjà chargé en section 5)
if 'model' not in dir() or model is None:
    tokenizer = AutoTokenizer.from_pretrained('outputs/checkpoints/best_model')
    model     = AutoModelForSeq2SeqLM.from_pretrained('outputs/checkpoints/best_model')
    device    = 'cuda' if torch.cuda.is_available() else 'cpu'
    model     = model.to(device).eval()

def translate(text: str, direction: str = 'fra-ewe') -> str:
    lang_map = {'ewe-fra': ('ewe_Latn', 'fra_Latn'),
                'fra-ewe': ('fra_Latn', 'ewe_Latn')}
    src_lang, tgt_lang = lang_map[direction]
    tokenizer.src_lang = src_lang
    inputs = tokenizer(text, return_tensors='pt',
                       max_length=128, truncation=True).to(device)
    forced_bos = tokenizer.convert_tokens_to_ids(tgt_lang)
    with torch.no_grad():
        out = model.generate(**inputs, forced_bos_token_id=forced_bos,
                             num_beams=4, max_length=256)
    return tokenizer.decode(out[0], skip_special_tokens=True).strip()

# ── Modifie ici ──────────────────────────────────────────────────────────────
TEXTE     = "L'eau potable est un droit fondamental pour toute personne."
DIRECTION = "fra-ewe"   # 'fra-ewe' ou 'ewe-fra'
# ─────────────────────────────────────────────────────────────────────────────

traduction = translate(TEXTE, DIRECTION)

print(f"Direction  : {DIRECTION}")
print(f"Source     : {TEXTE}")
print(f"Traduction : {traduction}")

In [ ]:
# ── 6.2 Lancement de l'interface Gradio ──────────────────────────────────────
#
# Génère un lien public temporaire (share=True) accessible depuis
# n'importe quel navigateur pendant la session Colab.
#
!python app.py